# Phase 8: ST000990 Raw Data Validation

**Version: 1.0** | 2026-03-11

**Question:** Does our elution sequence model generalize to independently-processed raw data from the same instrument platform?

| | Training Data | ST000990 |
|---|---|---|
| **Source** | 4 clinical cohorts (342 samples) | GLA lipid study (153 samples) |
| **Instrument** | SCIEX TripleTOF 6600+ | SCIEX TripleTOF 6600 (same) |
| **Column** | Waters CSH C18 | CSH C18 (same) |
| **Processing** | MS-DIAL (shared alignment) | MS-DIAL (independent, from raw .wiff) |
| **Features** | 15,242 across cohorts | 3,604 (merged from 3 batches) |

**Prerequisites:** Trained LSTM checkpoint from `01_train_models/outputs/lstm_best.pt`.

**Resume logic:** If Colab disconnects, re-run all cells. The evaluation cell reads `outputs/st000990_progress.csv` and skips completed samples.

**Changelog:**
- v1.0: Initial version

## 1. Setup

In [ ]:
import subprocess, sys, os, time, datetime

import torch
import torch.nn as nn
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# === Paths ===
BASE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{BASE_DIR}/08_st000990_validation"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model checkpoint
TRAIN_DIR = f"{BASE_DIR}/01_train_models"
LSTM_CHECKPOINT = f"{TRAIN_DIR}/outputs/lstm_best.pt"

# Data
ST000990_PATH = f"{EXPERIMENT_DIR}/merged_features.parquet"

# Checkpoint files
PROGRESS_CSV = f"{OUTPUT_DIR}/st000990_progress.csv"
LOG_CSV = f"{OUTPUT_DIR}/st000990_log.csv"

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"LSTM checkpoint: {LSTM_CHECKPOINT}")
print(f"Data: {ST000990_PATH}")
print(f"Progress checkpoint: {PROGRESS_CSV}")

In [ ]:
# === Configuration (must match training notebook) ===
VERSION = "1.0"
RANDOM_SEED = 42
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1
MZ_BIN_WIDTH = 10  # Da
MASS_DEFECT_BINS = 20
RT_BIN_WIDTH = 3   # seconds

RT_GAP_BINS = [-0.001, 0.1, 0.5, 1.0, 2.0, 5.0, 15.0, 9999]
RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
INTENSITY_RANK_BINS = [-0.001, 0.01, 0.05, 0.20, 0.50, 1.001]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print(f"Version: {VERSION}")

## 2. Model Definition & Checkpoint Loading

In [ ]:
class MultiFieldEmbedding(nn.Module):
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity):
        return (self.mz_embed(mz) + self.md_embed(md) + self.gap_embed(gap)
                + self.pol_embed(pol) + self.int_embed(intensity))


class LSTMModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_mz_classes)

    def forward(self, mz, md, gap, pol, intensity, hidden=None):
        x = self.embedding(mz, md, gap, pol, intensity)
        out, hidden = self.lstm(x, hidden)
        return self.head(self.dropout(out[:, -1, :])), hidden

In [ ]:
# === Load trained LSTM checkpoint ===
lstm_ckpt = torch.load(LSTM_CHECKPOINT, map_location=device, weights_only=False)
MAX_MZ_BIN = lstm_ckpt["config"]["num_classes"]
print(f"Max m/z bin: {MAX_MZ_BIN}")

embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7,
                max_polarity=3, max_intensity=5)

lstm_model = LSTMModel(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS,
                       DROPOUT, **embed_kw)
lstm_model.load_state_dict(lstm_ckpt["model_state_dict"])
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f"LSTM loaded -- training test: top1={lstm_ckpt['test_metrics']['top1']:.4f}, "
      f"MAE={lstm_ckpt['test_metrics']['mae_da']:.1f} Da")

## 3. Load & Tokenize ST000990

In [ ]:
# === Load merged feature table ===
feat = pd.read_parquet(ST000990_PATH)
print(f"Merged features: {len(feat)} features x {feat.shape[1]} columns")

# Identify sample columns
sample_cols = [c for c in feat.columns if c.startswith("GLA_TT6_Lipids_")]
nist_cols = [c for c in sample_cols if "_NIST" in c]
qc_cols = [c for c in sample_cols if "_QC" in c]
rep_cols = [c for c in sample_cols if "_Rep" in c]
analytical_cols = [c for c in sample_cols if "_SO" in c]

print(f"Samples: {len(sample_cols)} total")
print(f"  NIST: {len(nist_cols)}, QC: {len(qc_cols)}, "
      f"Rep: {len(rep_cols)}, Analytical: {len(analytical_cols)}")

# NOTE: rt_sec column is actually in MINUTES (MS-DIAL mzTabM mislabels)
# Rename for clarity
feat = feat.rename(columns={"rt_sec": "rt_min"})
print(f"\nm/z range: {feat['mz'].min():.1f} - {feat['mz'].max():.1f} Da")
print(f"RT range: {feat['rt_min'].min():.2f} - {feat['rt_min'].max():.2f} min")

In [ ]:
# === Convert wide format to per-sample long format + tokenize ===

def classify_sample(name):
    """Classify sample type from column name."""
    if "_NIST" in name:
        return "nist"
    elif "_QC" in name:
        return "qc"
    elif "_Rep" in name:
        return "replicate"
    else:
        return "analytical"


def tokenize_sample(feat_df, sample_col):
    """Extract detected features for one sample and tokenize."""
    # Get detected features (non-null abundance)
    mask = feat_df[sample_col].notna() & (feat_df[sample_col] > 0)
    sf = feat_df.loc[mask, ["mz", "rt_min", "adduct_ion", sample_col]].copy()
    sf = sf.rename(columns={sample_col: "intensity"})

    if len(sf) < CONTEXT_LENGTH + 1:
        return None  # Too few features for sliding window

    # Sort by RT (elution order)
    sf = sf.sort_values(["rt_min", "mz"]).reset_index(drop=True)

    # m/z bin
    sf["mz_bin"] = (sf["mz"] // MZ_BIN_WIDTH).astype(int).clip(upper=MAX_MZ_BIN - 1)

    # Mass defect bin
    mass_defect = sf["mz"] - np.floor(sf["mz"])
    sf["md_bin"] = (mass_defect * MASS_DEFECT_BINS).astype(int).clip(upper=MASS_DEFECT_BINS - 1)

    # RT gap (in seconds)
    sf["rt_gap"] = sf["rt_min"].diff() * 60  # min -> seconds
    sf["rt_gap"] = sf["rt_gap"].fillna(0).clip(lower=0)
    sf["rt_gap_idx"] = pd.cut(
        sf["rt_gap"], bins=RT_GAP_BINS, labels=RT_GAP_LABELS,
        right=False, include_lowest=True
    ).astype(str).map(RT_GAP_MAP).fillna(0).astype(int)

    # Polarity (all positive for ST000990)
    sf["polarity_idx"] = POLARITY_MAP["pos"]

    # Intensity rank
    pct_rank = 1 - sf["intensity"].rank(pct=True)
    sf["intensity_idx"] = pd.cut(
        pct_rank, bins=INTENSITY_RANK_BINS, labels=INTENSITY_RANK_LABELS,
        right=True, include_lowest=True
    ).astype(str).map(INTENSITY_MAP).fillna(4).astype(int)

    # Sequence position
    sf["seq_pos"] = range(len(sf))

    return sf


# Tokenize all samples
print("Tokenizing samples...")
sample_data = {}
skipped = 0
for col in sample_cols:
    result = tokenize_sample(feat, col)
    if result is not None:
        sample_data[col] = result
    else:
        skipped += 1

print(f"Tokenized: {len(sample_data)} samples ({skipped} skipped -- too few features)")
sizes = [len(v) for v in sample_data.values()]
print(f"Features per sample: min={min(sizes)}, median={np.median(sizes):.0f}, max={max(sizes)}")

# OOV check
all_mz_bins = pd.concat([v["mz_bin"] for v in sample_data.values()])
oov = (all_mz_bins >= MAX_MZ_BIN).sum()
print(f"Out-of-vocabulary m/z bins: {oov}/{len(all_mz_bins)} ({oov/len(all_mz_bins)*100:.1f}%)")

## 4. Evaluate Model (with checkpoint/restart)

In [ ]:
def evaluate_sample(model, sf, context_length=CONTEXT_LENGTH, top_k=(1, 3, 5, 10)):
    """Evaluate model on a single sample's tokenized features."""
    model.eval()
    results = []
    n = len(sf)

    mz_arr = sf["mz_bin"].values.astype(np.int64)
    md_arr = sf["md_bin"].values.astype(np.int64)
    gap_arr = sf["rt_gap_idx"].values.astype(np.int64)
    pol_arr = sf["polarity_idx"].values.astype(np.int64)
    int_arr = sf["intensity_idx"].values.astype(np.int64)

    with torch.no_grad():
        for pos in range(context_length, n):
            start = pos - context_length
            mz = torch.tensor(mz_arr[start:pos], dtype=torch.long).unsqueeze(0).to(device)
            md = torch.tensor(md_arr[start:pos], dtype=torch.long).unsqueeze(0).to(device)
            gap = torch.tensor(gap_arr[start:pos], dtype=torch.long).unsqueeze(0).to(device)
            pol = torch.tensor(pol_arr[start:pos], dtype=torch.long).unsqueeze(0).to(device)
            inten = torch.tensor(int_arr[start:pos], dtype=torch.long).unsqueeze(0).to(device)
            target = int(mz_arr[pos])

            logits, _ = model(mz, md, gap, pol, inten)
            pred = logits.argmax(dim=1).item()

            topk_hits = {}
            for k in top_k:
                _, tk = logits.topk(k, dim=1)
                topk_hits[f"top{k}"] = int(target in tk[0].cpu().numpy())

            results.append({
                "position": pos,
                "target": target,
                "pred": pred,
                "mae_da": abs(pred - target) * MZ_BIN_WIDTH,
                **topk_hits,
            })

    return results

In [ ]:
# === Main evaluation loop with checkpoint/restart ===

# Load existing progress if resuming
completed_samples = set()
all_results = []
if os.path.exists(PROGRESS_CSV):
    progress_df = pd.read_csv(PROGRESS_CSV)
    completed_samples = set(progress_df["sample_id"].unique())
    # Reload detailed results
    detail_path = f"{OUTPUT_DIR}/st000990_detailed_partial.parquet"
    if os.path.exists(detail_path):
        all_results = pd.read_parquet(detail_path).to_dict("records")
    print(f"RESUMING: {len(completed_samples)} samples already done, "
          f"{len(sample_data) - len(completed_samples)} remaining")
else:
    print(f"Starting fresh: {len(sample_data)} samples to evaluate")

# Initialize log file
if not os.path.exists(LOG_CSV):
    with open(LOG_CSV, "w") as f:
        f.write("timestamp,sample_id,sample_type,n_features,n_predictions,"
                "top1,top5,mae_da,elapsed_s\n")

# Evaluate remaining samples
remaining = [(name, sf) for name, sf in sample_data.items()
             if name not in completed_samples]
total = len(sample_data)
done = len(completed_samples)
t0 = time.time()

for i, (sample_name, sf) in enumerate(remaining):
    t_sample = time.time()
    sample_type = classify_sample(sample_name)

    # Evaluate
    results = evaluate_sample(lstm_model, sf)

    if not results:
        continue

    # Tag with sample info
    for r in results:
        r["sample_id"] = sample_name
        r["sample_type"] = sample_type
    all_results.extend(results)

    # Compute sample-level metrics
    rdf = pd.DataFrame(results)
    sample_top1 = rdf["top1"].mean()
    sample_top5 = rdf["top5"].mean()
    sample_mae = rdf["mae_da"].mean()
    elapsed = time.time() - t_sample
    done += 1

    # Log to CSV (append)
    ts = datetime.datetime.now().isoformat()
    with open(LOG_CSV, "a") as f:
        f.write(f"{ts},{sample_name},{sample_type},{len(sf)},{len(results)},"
                f"{sample_top1:.6f},{sample_top5:.6f},{sample_mae:.1f},{elapsed:.1f}\n")

    # Save progress checkpoint every 10 samples
    if done % 10 == 0 or i == len(remaining) - 1:
        # Save progress CSV
        progress_records = []
        rdf_all = pd.DataFrame(all_results)
        for sid in rdf_all["sample_id"].unique():
            s = rdf_all[rdf_all["sample_id"] == sid]
            progress_records.append({
                "sample_id": sid,
                "sample_type": s["sample_type"].iloc[0],
                "n_predictions": len(s),
                "top1": s["top1"].mean(),
                "top5": s["top5"].mean(),
                "mae_da": s["mae_da"].mean(),
            })
        pd.DataFrame(progress_records).to_csv(PROGRESS_CSV, index=False)

        # Save partial detailed results
        pd.DataFrame(all_results).to_parquet(
            f"{OUTPUT_DIR}/st000990_detailed_partial.parquet", index=False)

    # Print progress
    total_elapsed = time.time() - t0
    rate = done / max(total_elapsed, 1) if i > 0 else 0
    eta = (total - done) / rate / 60 if rate > 0 else 0
    print(f"  [{done}/{total}] {sample_name} ({sample_type}): "
          f"top1={sample_top1:.4f} top5={sample_top5:.4f} MAE={sample_mae:.0f}Da "
          f"({len(results)} preds, {elapsed:.1f}s) ETA: {eta:.0f}min")

print(f"\nDone! {done}/{total} samples evaluated in {(time.time()-t0)/60:.1f} min")

## 5. Aggregate Results

In [ ]:
# === Aggregate metrics ===
results_df = pd.DataFrame(all_results)
print(f"Total predictions: {len(results_df):,}")
print(f"Unique samples: {results_df['sample_id'].nunique()}")

# Overall metrics
overall = {
    "top1": results_df["top1"].mean(),
    "top3": results_df["top3"].mean(),
    "top5": results_df["top5"].mean(),
    "top10": results_df["top10"].mean(),
    "mae_da": results_df["mae_da"].mean(),
    "n": len(results_df),
}

# Per sample-type metrics
type_metrics = {}
for stype in results_df["sample_type"].unique():
    mask = results_df["sample_type"] == stype
    sub = results_df[mask]
    type_metrics[stype] = {
        "top1": sub["top1"].mean(),
        "top5": sub["top5"].mean(),
        "mae_da": sub["mae_da"].mean(),
        "n_samples": sub["sample_id"].nunique(),
        "n_predictions": len(sub),
    }

# Compare with training performance
train_top1 = lstm_ckpt["test_metrics"]["top1"]
train_top5 = lstm_ckpt["test_metrics"]["top5"]
train_mae = lstm_ckpt["test_metrics"]["mae_da"]

print(f"\n{'='*70}")
print(f"{'Dataset':<30} {'Top-1':<10} {'Top-5':<10} {'MAE(Da)':<10} {'n'}")
print(f"{'-'*70}")
print(f"{'Training cohorts (test)':<30} {train_top1:<10.4f} {train_top5:<10.4f} {train_mae:<10.1f} {lstm_ckpt['test_metrics']['n']}")
print(f"{'ST000990 (all)':<30} {overall['top1']:<10.4f} {overall['top5']:<10.4f} {overall['mae_da']:<10.1f} {overall['n']}")
for stype, m in sorted(type_metrics.items()):
    label = f"  ST000990 ({stype})"
    print(f"{label:<30} {m['top1']:<10.4f} {m['top5']:<10.4f} {m['mae_da']:<10.1f} {m['n_predictions']}")
print(f"{'Random baseline':<30} {'0.0109':<10} {'':<10} {'1965':<10}")
print(f"{'='*70}")

gen_ratio = overall["top1"] / train_top1 if train_top1 > 0 else 0
print(f"\nGeneralization ratio: {gen_ratio:.1%} of training-set performance retained")

# Compare with ST003514 (Phase 6)
phase6_path = f"{BASE_DIR}/02_external_validation/outputs/phase6_results.json"
if os.path.exists(phase6_path):
    with open(phase6_path) as f:
        p6 = json.load(f)
    st3514_top1 = p6["lstm"]["external"]["top1"]
    print(f"ST003514 (different instrument): {st3514_top1:.4f} top-1 ({st3514_top1/train_top1:.1%} retained)")
    print(f"ST000990 (same instrument):      {overall['top1']:.4f} top-1 ({gen_ratio:.1%} retained)")
    improvement = overall["top1"] / st3514_top1 if st3514_top1 > 0 else float("inf")
    print(f"Improvement factor: {improvement:.1f}x")

## 6. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel A: Cross-dataset accuracy comparison ---
ax = axes[0, 0]
datasets = ["Training\n(test set)", "ST000990\n(same instr.)", "ST003514\n(diff instr.)"]
top1_vals = [train_top1, overall["top1"], 0]
top5_vals = [train_top5, overall["top5"], 0]
# Try to load ST003514 results
if os.path.exists(phase6_path):
    with open(phase6_path) as f:
        p6 = json.load(f)
    top1_vals[2] = p6["lstm"]["external"]["top1"]
    top5_vals[2] = p6["lstm"]["external"]["top5"]

x = np.arange(len(datasets))
w = 0.35
ax.bar(x - w/2, top1_vals, w, label="Top-1", color="tab:blue")
ax.bar(x + w/2, top5_vals, w, label="Top-5", color="tab:orange")
for i, (v1, v5) in enumerate(zip(top1_vals, top5_vals)):
    ax.text(i - w/2, v1 + 0.01, f"{v1:.1%}", ha="center", fontsize=8)
    ax.text(i + w/2, v5 + 0.01, f"{v5:.1%}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylabel("Accuracy")
ax.set_title("A. Cross-Dataset Generalization")
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)

# --- Panel B: Per-sample top-1 accuracy distribution ---
ax = axes[0, 1]
sample_acc = results_df.groupby(["sample_id", "sample_type"])["top1"].mean().reset_index()
colors = {"nist": "tab:red", "qc": "tab:green", "replicate": "tab:blue", "analytical": "tab:gray"}
for stype in ["analytical", "replicate", "qc", "nist"]:
    mask = sample_acc["sample_type"] == stype
    if mask.sum() > 0:
        ax.hist(sample_acc.loc[mask, "top1"], bins=20, alpha=0.6,
                label=f"{stype} (n={mask.sum()})", color=colors.get(stype, "gray"))
ax.set_xlabel("Top-1 Accuracy")
ax.set_ylabel("Number of Samples")
ax.set_title("B. Per-Sample Accuracy Distribution")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Panel C: Accuracy by position ---
ax = axes[1, 0]
results_df["pos_bin"] = (results_df["position"] // 50) * 50
pos_acc = results_df.groupby("pos_bin")["top1"].agg(["mean", "std", "count"])
pos_acc = pos_acc[pos_acc["count"] >= 10]  # filter sparse bins
ax.plot(pos_acc.index, pos_acc["mean"], color="tab:blue", linewidth=2)
ax.fill_between(pos_acc.index,
                pos_acc["mean"] - pos_acc["std"],
                pos_acc["mean"] + pos_acc["std"],
                alpha=0.2, color="tab:blue")
ax.axhline(overall["top1"], color="gray", linestyle="--", alpha=0.5, label=f"Overall: {overall['top1']:.1%}")
ax.set_xlabel("Position in Elution Sequence")
ax.set_ylabel("Top-1 Accuracy")
ax.set_title("C. Accuracy vs Sequence Position")
ax.legend()
ax.grid(alpha=0.3)

# --- Panel D: Per sample-type bar chart ---
ax = axes[1, 1]
types = sorted(type_metrics.keys())
x = np.arange(len(types))
t1 = [type_metrics[t]["top1"] for t in types]
t5 = [type_metrics[t]["top5"] for t in types]
w = 0.35
ax.bar(x - w/2, t1, w, label="Top-1", color="tab:blue")
ax.bar(x + w/2, t5, w, label="Top-5", color="tab:orange")
for i, (v1, v5) in enumerate(zip(t1, t5)):
    ax.text(i - w/2, v1 + 0.01, f"{v1:.1%}", ha="center", fontsize=8)
    ax.text(i + w/2, v5 + 0.01, f"{v5:.1%}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([t.capitalize() for t in types])
ax.set_ylabel("Accuracy")
ax.set_title("D. Accuracy by Sample Type")
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/st000990_validation.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.savefig(fig_path.replace(".png", ".pdf"), bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 7. Save Final Results

In [ ]:
# === Save final results ===
final_results = {
    "experiment": "Phase 8: ST000990 Raw Data Validation",
    "version": VERSION,
    "dataset": "ST000990 (GLA lipid study)",
    "instrument": "SCIEX TripleTOF 6600 (same as training)",
    "column": "CSH C18 (same as training)",
    "processing": "MS-DIAL from raw .wiff files (3 batches, post-hoc merged)",
    "n_features": len(feat),
    "n_samples_evaluated": results_df["sample_id"].nunique(),
    "n_predictions": len(results_df),
    "overall_metrics": overall,
    "per_type_metrics": type_metrics,
    "training_test_metrics": lstm_ckpt["test_metrics"],
    "generalization_ratio": gen_ratio,
}

# Add ST003514 comparison if available
if os.path.exists(phase6_path):
    with open(phase6_path) as f:
        p6 = json.load(f)
    final_results["st003514_comparison"] = {
        "st003514_top1": p6["lstm"]["external"]["top1"],
        "st000990_top1": overall["top1"],
        "improvement_factor": overall["top1"] / p6["lstm"]["external"]["top1"] if p6["lstm"]["external"]["top1"] > 0 else None,
    }

with open(f"{OUTPUT_DIR}/st000990_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)

# Save detailed results (final)
results_df.to_parquet(f"{OUTPUT_DIR}/st000990_detailed.parquet", index=False)

# Clean up partial checkpoint
partial_path = f"{OUTPUT_DIR}/st000990_detailed_partial.parquet"
if os.path.exists(partial_path):
    os.remove(partial_path)

print("All results saved!")
print(f"Output directory: {OUTPUT_DIR}")
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f_name}")
    print(f"  {f_name:<45} {size/1024:.0f} KB")